<a href="https://colab.research.google.com/github/AmitabhTS/AML-LAB-EXPERIMENTS/blob/main/AML7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

car_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/AML/car_price_cleaned.csv")
target_car = "price"       # Change according to the actual dataset

# Clean the target variable 'price'
car_df[target_car] = car_df[target_car].replace({r'\$': '', r',': ''}, regex=True).astype(float)

# Feature engineering
current_year = pd.Timestamp.now().year

if "year" in car_df.columns:
    car_df["vehicle_age"] = current_year - car_df["year"]

if "km_driven" in car_df.columns and "vehicle_age" in car_df.columns:
    car_df["km_per_year"] = (
        car_df["km_driven"] / (car_df["vehicle_age"] + 1)
    )

# Remove target and rows with missing target
car_df = car_df.dropna(subset=[target_car])

X = car_df.drop(target_car, axis=1)
y = car_df[target_car]

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("numerical", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(
        random_state=42,
        max_depth=12
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    )
}

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)
car_predictions = {}

for name, regressor in models.items():
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", regressor)
    ])
    model.fit(X_train, y_train)
    car_predictions[name] = model.predict(X_test)
    print(name, "training completed")

Linear Regression training completed
Decision Tree training completed
Random Forest training completed
Gradient Boosting training completed
